# Testing Feature engineering first try
No son horas - Andres Calamaro

## 1. FUNCTIONS

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
 
from scipy.stats import kurtosis, skew
from scipy.signal import welch
 
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
 

In [2]:
# 1. Configuration 

WINDOW_SEC  = 10          # sec for windows
OVERLAP     = 0.25        # overlap (25%)
STEP_SEC    = WINDOW_SEC * (1 - OVERLAP)   # 7.5 s
 
BANDS = {
    "delta": (0.5, 4),
    "theta": (4,   8),
    "alpha": (8,  13),
    "beta":  (13, 30),
    "gamma": (30, 45),
}
 

In [3]:
def parse_iso(ts) -> float:
    """ISO string → Unix timestamp (float seconds)."""
    dt = datetime.fromisoformat(str(ts))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.timestamp()
 
 
def load_npz(filepath: str) -> dict:
    """
    Carga el .npz y devuelve un dict con todo parseado.
 
    Retorna:
        {
            "X":             np.ndarray (C, N),
            "mu":            np.ndarray (C,),
            "sigma":         np.ndarray (C,),
            "fs":            float,
            "channel_names": list[str],
            "seizure_onsets_unix": list[float],   # timestamps en segundos Unix
            "t0_unix":       float,               # inicio grabación
        }
    """
    data = np.load(filepath, allow_pickle=True)
 
    X  = data["X"]                          # (C, N)
    fs = float(data["fs"])
 
    channel_names = [
        c.decode() if isinstance(c, (bytes, np.bytes_)) else str(c)
        for c in data["channel_names"]
    ]
 
    t0_unix = parse_iso(data["T0"][0])
 
    seizure_onsets_unix = [parse_iso(ts) for ts in data["seizure_onsets"]]
 
    print(f"Cargado: {filepath}")
    print(f"  EEG shape : {X.shape}  (canales={X.shape[0]}, samples={X.shape[1]})")
    print(f"  fs        : {fs} Hz")
    print(f"  Canales   : {channel_names}")
    print(f"  Seizures  : {len(seizure_onsets_unix)}")
    print(f"  Duración  : {X.shape[1]/fs/60:.1f} min")
 
    return {
        "X": X,
        "mu": data["mu"],
        "sigma": data["sigma"],
        "fs": fs,
        "channel_names": channel_names,
        "seizure_onsets_unix": seizure_onsets_unix,
        "t0_unix": t0_unix,
    }
 
 

In [4]:
# 3. EPOCHING with windowing


def epoch_eeg(rec: dict,
              window_sec: float = WINDOW_SEC,
              step_sec: float   = STEP_SEC) -> tuple[np.ndarray, np.ndarray]:
    """
    split EEG continous overlap.
 
    label = 1 si algún seizure_onset cae dentro de [t_start, t_end).
 
    Return:
        epochs → (n_epochs, C, n_samples)
        labels → (n_epochs,)  int  0/1
    """
    X       = rec["X"]
    fs      = rec["fs"]
    t0      = rec["t0_unix"]
    onsets  = rec["seizure_onsets_unix"]
 
    win_samples  = int(window_sec * fs)
    step_samples = int(step_sec   * fs)
    n_total      = X.shape[1]
 
    epochs, labels = [], []
 
    start = 0
    while start + win_samples <= n_total:
        end = start + win_samples
 
        # Tiempo absoluto de la ventana
        t_start = t0 + start / fs
        t_end   = t0 + end   / fs
 
        # Etiqueta: 1 si algún onset cae dentro de la ventana
        label = int(any(t_start <= onset < t_end for onset in onsets))
 
        epochs.append(X[:, start:end])   # (C, win_samples)
        labels.append(label)
 
        start += step_samples
 
    epochs = np.array(epochs)   # (n_epochs, C, win_samples)
    labels = np.array(labels)   # (n_epochs,)
 
    n_seiz = labels.sum()
    print(f"\nEpoching completo:")
    print(f"  Total ventanas : {len(labels)}")
    print(f"  Seizure        : {n_seiz}  ({100*n_seiz/len(labels):.1f}%)")
    print(f"  No-seizure     : {len(labels)-n_seiz}")
 
    return epochs, labels
 

In [6]:
# 4. FEATURE EXTRACTION

def band_power(signal: np.ndarray, fs: float, fmin: float, fmax: float) -> float:
    """Absolute band power using Welch's method."""
    nperseg = min(len(signal), int(fs * 2))
    freqs, psd = welch(signal, fs=fs, nperseg=nperseg)
    idx = (freqs >= fmin) & (freqs <= fmax)
    return float(np.trapz(psd[idx], freqs[idx]))
 
 
def extract_features_epoch(epoch: np.ndarray,
                            fs: float,
                            channel_names: list[str]) -> dict:
    """
    Extract features from one epoch of shape (C, n_samples).
 
    Per channel:
        Statistical : mean, var, std, kurtosis, skewness, peak-to-peak
        Band power  : delta, theta, alpha, beta, gamma
    """
    features = {}
 
    for ch_idx, ch_name in enumerate(channel_names):
        sig    = epoch[ch_idx]
        prefix = ch_name.replace(" ", "_")
 
        # Statistical features
        features[f"{prefix}_mean"]      = float(np.mean(sig))
        features[f"{prefix}_var"]       = float(np.var(sig))
        features[f"{prefix}_std"]       = float(np.std(sig))
        features[f"{prefix}_kurtosis"]  = float(kurtosis(sig))
        features[f"{prefix}_skewness"]  = float(skew(sig))
        features[f"{prefix}_peak2peak"] = float(np.ptp(sig))
 
        # Band power features
        for band_name, (fmin, fmax) in BANDS.items():
            features[f"{prefix}_{band_name}"] = band_power(sig, fs, fmin, fmax)
 
    return features
 
 
def build_feature_dataframe(epochs: np.ndarray,
                             labels: np.ndarray,
                             fs: float,
                             channel_names: list[str]) -> pd.DataFrame:
    """
    Build the in-memory feature DataFrame.
    Rows = epochs, columns = features + 'label'.
    """
    rows = [
        extract_features_epoch(epoch, fs, channel_names)
        for epoch in epochs
    ]
    df = pd.DataFrame(rows)
    df["label"] = labels.astype(int)
 
    print(f"\nFeature DataFrame: {df.shape}")
    print(f"  Features per epoch : {df.shape[1] - 1}")
    return df
 

In [7]:
# 5. SVM MODEL
def build_pipeline() -> Pipeline:
    """
    StandardScaler → SVC (RBF kernel).
    class_weight='balanced' handles the heavy seizure/no-seizure imbalance.
    probability=True enables ROC-AUC scoring.
    """
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42,
        )),
    ])

In [8]:
# 6. EVALUATION
def evaluate_cv(df: pd.DataFrame, n_splits: int = 5) -> dict:
    """
    Stratified K-Fold cross-validation.
    Metrics: balanced_accuracy, f1, roc_auc, recall, precision.
 
    NOTE: with continuous EEG data, consider GroupKFold grouped by
    recording to avoid data leakage between overlapping windows.
    """
    X = df.drop(columns="label").values
    y = df["label"].values
 
    cv      = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scoring = ["balanced_accuracy", "f1", "roc_auc", "recall", "precision"]
 
    results = cross_validate(
        build_pipeline(), X, y,
        cv=cv, scoring=scoring,
        return_train_score=False,
        n_jobs=-1,
    )
 
    print("\n── Cross-Validation Results ─────────────────────────")
    for metric in scoring:
        scores = results[f"test_{metric}"]
        print(f"  {metric:22s}: {scores.mean():.3f} ± {scores.std():.3f}")
    print("─────────────────────────────────────────────────────\n")
    return results
 
 
def tune_hyperparameters(df: pd.DataFrame) -> Pipeline:
    """
    GridSearchCV over C and gamma, optimising ROC-AUC.
    """
    X = df.drop(columns="label").values
    y = df["label"].values
 
    param_grid = {
        "svm__C":     [0.1, 1, 10, 100],
        "svm__gamma": ["scale", "auto", 0.001, 0.01],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
    search = GridSearchCV(
        build_pipeline(), param_grid,
        scoring="roc_auc", cv=cv,
        n_jobs=-1, verbose=1,
    )
    search.fit(X, y)
 
    print(f"\nBest parameters : {search.best_params_}")
    print(f"Best ROC-AUC    : {search.best_score_:.3f}\n")
    return search.best_estimator_
 
 

In [9]:
#7. MAIN
if __name__ == "__main__":
 
    NPZ_PATH = "data/recording.npz"   # ← update to your path
 
    # Step 1 — Load NPZ
    rec = load_npz(NPZ_PATH)
 
    # Step 2 — Epoching
    epochs, labels = epoch_eeg(rec)
 
    # Step 3 — Feature extraction → DataFrame
    df_features = build_feature_dataframe(
        epochs, labels,
        fs=rec["fs"],
        channel_names=rec["channel_names"],
    )
 
    # Optional: persist features to avoid recomputing
    # df_features.to_parquet("features.parquet", index=False)
    # df_features = pd.read_parquet("features.parquet")
 
    # Step 4 — Cross-validation evaluation
    cv_results = evaluate_cv(df_features, n_splits=5)
 
    # Step 5 (optional) — Hyperparameter tuning
    # best_model = tune_hyperparameters(df_features)
 
    # Step 6 — Final model trained on all data
    pipeline = build_pipeline()
    X_all = df_features.drop(columns="label").values
    y_all = df_features["label"].values
    pipeline.fit(X_all, y_all)
 
    y_pred = pipeline.predict(X_all)
    y_prob = pipeline.predict_proba(X_all)[:, 1]
 
    print("── Full-dataset report (reference only, not validation) ──")
    print(classification_report(y_all, y_pred,
                                 target_names=["no-seizure", "seizure"]))
    print(f"ROC-AUC          : {roc_auc_score(y_all, y_prob):.3f}")
    print(f"Confusion matrix :\n{confusion_matrix(y_all, y_pred)}")
 

FileNotFoundError: [Errno 2] No such file or directory: 'data/recording.npz'

## 2. TESTING 


In [10]:
import numpy as np
NPZ_PATH = "/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/results/XB47Y_test2702204/XB47Y_181_zscore_full.npz"  

data = np.load(NPZ_PATH, allow_pickle=True)

print("Keys en el .npz:")
for key in data.files:
    val = data[key]
    print(f"  {key:20s} | dtype: {str(val.dtype):12s} | shape: {val.shape} | ejemplo: {val.flat[0]!r}")

Keys en el .npz:
  X                    | dtype: float32      | shape: (2, 4472028) | ejemplo: np.float32(-0.27410796)
  mu                   | dtype: float32      | shape: (2,) | ejemplo: np.float32(0.0008241759)
  sigma                | dtype: float32      | shape: (2,) | ejemplo: np.float32(65.476364)
  fs                   | dtype: float64      | shape: () | ejemplo: np.float64(207.031054664947)
  channel_names        | dtype: object       | shape: (2,) | ejemplo: 'EEG SQ_D-SQ_C'
  source_file          | dtype: object       | shape: (1,) | ejemplo: 'XB47Y_181.mat'
  seizure_onsets       | dtype: object       | shape: (5,) | ejemplo: '2019-12-11 10:02:37.652000'
  T0                   | dtype: object       | shape: (5,) | ejemplo: '2019-12-11 10:00:59.000000'
  TF                   | dtype: object       | shape: (5,) | ejemplo: '2019-12-11 16:00:59.759400'


In [11]:
import numpy as np
from datetime import datetime, timezone

data = np.load(NPZ_PATH, allow_pickle=True)

fs = float(data["fs"])
X  = data["X"]

T0s = data["T0"]
TFs = data["TF"]
onsets = data["seizure_onsets"]

print(f"fs       : {fs:.4f} Hz")
print(f"X shape  : {X.shape}  →  duración total: {X.shape[1]/fs/60:.1f} min")
print(f"\nSegmentos (T0 / TF):")
for i, (t0, tf) in enumerate(zip(T0s, TFs)):
    dt0 = datetime.fromisoformat(str(t0))
    dtf = datetime.fromisoformat(str(tf))
    dur = (dtf - dt0).total_seconds()
    print(f"  [{i}]  {t0}  →  {tf}  ({dur/60:.1f} min)")

print(f"\nSeizure onsets ({len(onsets)}):")
for i, onset in enumerate(onsets):
    print(f"  [{i}]  {onset}")

# Verificar si la duración total de segmentos coincide con X
total_seg_sec = sum(
    (datetime.fromisoformat(str(tf)) - datetime.fromisoformat(str(t0))).total_seconds()
    for t0, tf in zip(T0s, TFs)
)
total_X_sec = X.shape[1] / fs
print(f"\nDuración total segmentos : {total_seg_sec/60:.1f} min")
print(f"Duración total X         : {total_X_sec/60:.1f} min")
print(f"Diferencia               : {abs(total_seg_sec - total_X_sec):.1f} s")

fs       : 207.0311 Hz
X shape  : (2, 4472028)  →  duración total: 360.0 min

Segmentos (T0 / TF):
  [0]  2019-12-11 10:00:59.000000  →  2019-12-11 16:00:59.759400  (360.0 min)
  [1]  2019-12-11 10:00:59.000000  →  2019-12-11 16:00:59.759400  (360.0 min)
  [2]  2019-12-11 10:00:59.000000  →  2019-12-11 16:00:59.759400  (360.0 min)
  [3]  2019-12-11 10:00:59.000000  →  2019-12-11 16:00:59.759400  (360.0 min)
  [4]  2019-12-11 10:00:59.000000  →  2019-12-11 16:00:59.759400  (360.0 min)

Seizure onsets (5):
  [0]  2019-12-11 10:02:37.652000
  [1]  2019-12-11 10:26:56.951000
  [2]  2019-12-11 10:52:32.102000
  [3]  2019-12-11 14:48:10.356000
  [4]  2019-12-11 14:54:11.513000

Duración total segmentos : 1800.1 min
Duración total X         : 360.0 min
Diferencia               : 86403.0 s


In [12]:
# Una sola grabación continua
t0_unix = parse_iso(data["T0"][0])   # usar solo [0]
seizure_onsets_unix = [parse_iso(ts) for ts in data["seizure_onsets"]]

# Verificar que los onsets caen dentro de la grabación
t0 = datetime.fromisoformat(str(data["T0"][0]))
tf = datetime.fromisoformat(str(data["TF"][0]))
total_sec = X.shape[1] / fs

print(f"Grabación : {t0}  →  {tf}")
print(f"Duración  : {total_sec/60:.1f} min  ({total_sec:.1f} s)\n")

print("Seizure onsets — posición en la grabación:")
for i, (ts_raw, ts_unix) in enumerate(zip(data["seizure_onsets"], seizure_onsets_unix)):
    offset_sec = ts_unix - t0_unix
    offset_min = offset_sec / 60
    sample_idx = int(offset_sec * fs)
    in_range   = 0 <= sample_idx < X.shape[1]
    print(f"  [{i}]  +{offset_min:6.1f} min  →  sample {sample_idx:>8,}  {'✓' if in_range else '✗ OUT OF RANGE'}")

Grabación : 2019-12-11 10:00:59  →  2019-12-11 16:00:59.759400
Duración  : 360.0 min  (21600.8 s)

Seizure onsets — posición en la grabación:
  [0]  +   1.6 min  →  sample   20,424  ✓
  [1]  +  26.0 min  →  sample  322,544  ✓
  [2]  +  51.6 min  →  sample  640,368  ✓
  [3]  + 287.2 min  →  sample 3,567,425  ✓
  [4]  + 293.2 min  →  sample 3,642,196  ✓


In [13]:
win_samples  = int(WINDOW_SEC * fs)   # ~2070 samples
step_samples = int(STEP_SEC   * fs)   # ~1552 samples

print(f"Ventana : {WINDOW_SEC}s  →  {win_samples} samples")
print(f"Step    : {STEP_SEC}s   →  {step_samples} samples")

epochs, labels = [], []

start = 0
while start + win_samples <= X.shape[1]:
    end     = start + win_samples
    t_start = t0_unix + start / fs
    t_end   = t0_unix + end   / fs

    label = int(any(t_start <= onset < t_end for onset in seizure_onsets_unix))

    epochs.append(X[:, start:end])
    labels.append(label)
    start += step_samples

epochs = np.array(epochs)
labels = np.array(labels)

n_seiz = labels.sum()
print(f"\nEpoching completo:")
print(f"  Total ventanas : {len(labels):,}")
print(f"  Seizure        : {n_seiz}  ({100*n_seiz/len(labels):.2f}%)")
print(f"  No-seizure     : {len(labels)-n_seiz:,}")
print(f"  Shape epochs   : {epochs.shape}")

# Sanity check: ver en qué ventanas cayó cada onset
print(f"\nVentanas con seizure (índices):")
seizure_indices = np.where(labels == 1)[0]
for idx in seizure_indices:
    t_s = t0_unix + idx * step_samples / fs
    t_e = t_s + WINDOW_SEC
    print(f"  ventana [{idx:,}]  →  offset {(t_s - t0_unix)/60:.2f} – {(t_e - t0_unix)/60:.2f} min")

Ventana : 10s  →  2070 samples
Step    : 7.5s   →  1552 samples

Epoching completo:
  Total ventanas : 2,881
  Seizure        : 6  (0.21%)
  No-seizure     : 2,875
  Shape epochs   : (2881, 2, 2070)

Ventanas con seizure (índices):
  ventana [12]  →  offset 1.50 – 1.67 min
  ventana [13]  →  offset 1.62 – 1.79 min
  ventana [207]  →  offset 25.86 – 26.03 min
  ventana [412]  →  offset 51.48 – 51.64 min
  ventana [2,298]  →  offset 287.11 – 287.28 min
  ventana [2,346]  →  offset 293.11 – 293.28 min


In [14]:
from scipy.stats import kurtosis, skew
from scipy.signal import welch

BANDS = {
    "delta": (0.5,  4),
    "theta": (4,    8),
    "alpha": (8,   13),
    "beta":  (13,  30),
    "gamma": (30,  45),
}

channel_names = [str(c) for c in data["channel_names"]]

def band_power(signal, fs, fmin, fmax):
    nperseg = min(len(signal), int(fs * 2))
    freqs, psd = welch(signal, fs=fs, nperseg=nperseg)
    idx = (freqs >= fmin) & (freqs <= fmax)
    return float(np.trapz(psd[idx], freqs[idx]))

def extract_features_epoch(epoch, fs, channel_names):
    """epoch shape: (C, n_samples)"""
    features = {}
    for ch_idx, ch_name in enumerate(channel_names):
        sig    = epoch[ch_idx]
        prefix = ch_name.replace(" ", "_")

        features[f"{prefix}_mean"]      = float(np.mean(sig))
        features[f"{prefix}_var"]       = float(np.var(sig))
        features[f"{prefix}_std"]       = float(np.std(sig))
        features[f"{prefix}_kurtosis"]  = float(kurtosis(sig))
        features[f"{prefix}_skewness"]  = float(skew(sig))
        features[f"{prefix}_peak2peak"] = float(np.ptp(sig))

        for band_name, (fmin, fmax) in BANDS.items():
            features[f"{prefix}_{band_name}"] = band_power(sig, fs, fmin, fmax)

    return features

# --- Probar en UNA ventana antes de correr todo ---
test_feat = extract_features_epoch(epochs[0], fs, channel_names)

print(f"Features extraídas de ventana [0]: {len(test_feat)}")
print(f"\nEjemplo de valores:")
for k, v in test_feat.items():
    print(f"  {k:45s}: {v:.6f}")

Features extraídas de ventana [0]: 22

Ejemplo de valores:
  EEG_SQ_D-SQ_C_mean                           : 0.042017
  EEG_SQ_D-SQ_C_var                            : 1.128411
  EEG_SQ_D-SQ_C_std                            : 1.062267
  EEG_SQ_D-SQ_C_kurtosis                       : 2.508635
  EEG_SQ_D-SQ_C_skewness                       : 1.242491
  EEG_SQ_D-SQ_C_peak2peak                      : 6.019582
  EEG_SQ_D-SQ_C_delta                          : 0.202016
  EEG_SQ_D-SQ_C_theta                          : 0.036808
  EEG_SQ_D-SQ_C_alpha                          : 0.010602
  EEG_SQ_D-SQ_C_beta                           : 0.029822
  EEG_SQ_D-SQ_C_gamma                          : 0.531475
  EEG_SQ_P-SQ_C_mean                           : 0.134252
  EEG_SQ_P-SQ_C_var                            : 13.947617
  EEG_SQ_P-SQ_C_std                            : 3.734651
  EEG_SQ_P-SQ_C_kurtosis                       : 2.539744
  EEG_SQ_P-SQ_C_skewness                       : 1.209874
  EEG_SQ_P-S

/tmp/ipykernel_2014674/2117566873.py:18: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[idx], freqs[idx]))


In [15]:
import pandas as pd
from scipy.stats import kurtosis, skew
from scipy.signal import welch

def band_power(signal, fs, fmin, fmax):
    nperseg = min(len(signal), int(fs * 2))
    freqs, psd = welch(signal, fs=fs, nperseg=nperseg)
    idx = (freqs >= fmin) & (freqs <= fmax)
    return float(np.trapezoid(psd[idx], freqs[idx]))  # ← fix

def extract_features_epoch(epoch, fs, channel_names):
    features = {}
    for ch_idx, ch_name in enumerate(channel_names):
        sig    = epoch[ch_idx]
        prefix = ch_name.replace(" ", "_")

        features[f"{prefix}_mean"]      = float(np.mean(sig))
        features[f"{prefix}_var"]       = float(np.var(sig))
        features[f"{prefix}_std"]       = float(np.std(sig))
        features[f"{prefix}_kurtosis"]  = float(kurtosis(sig))
        features[f"{prefix}_skewness"]  = float(skew(sig))
        features[f"{prefix}_peak2peak"] = float(np.ptp(sig))

        for band_name, (fmin, fmax) in BANDS.items():
            features[f"{prefix}_{band_name}"] = band_power(sig, fs, fmin, fmax)

    return features

# Procesar todas las ventanas (puede tardar ~30s)
print("Extracting features...")
rows = [extract_features_epoch(epoch, fs, channel_names) for epoch in epochs]

df = pd.DataFrame(rows)
df["label"] = labels.astype(int)

print(f"Done. DataFrame shape: {df.shape}")
print(f"\nClase distribution:")
print(df["label"].value_counts().rename({0: "no-seizure", 1: "seizure"}))
print(f"\nNaN check: {df.isna().sum().sum()} NaNs")
print(f"\nPrimeras filas:")
df.head(3)

Extracting features...
Done. DataFrame shape: (2881, 23)

Clase distribution:
label
no-seizure    2875
seizure          6
Name: count, dtype: int64

NaN check: 0 NaNs

Primeras filas:


,EEG_SQ_D-SQ_C_mean,EEG_SQ_D-SQ_C_var,EEG_SQ_D-SQ_C_std,EEG_SQ_D-SQ_C_kurtosis,EEG_SQ_D-SQ_C_skewness,EEG_SQ_D-SQ_C_peak2peak,EEG_SQ_D-SQ_C_delta,EEG_SQ_D-SQ_C_theta,EEG_SQ_D-SQ_C_alpha,EEG_SQ_D-SQ_C_beta,...,EEG_SQ_P-SQ_C_std,EEG_SQ_P-SQ_C_kurtosis,EEG_SQ_P-SQ_C_skewness,EEG_SQ_P-SQ_C_peak2peak,EEG_SQ_P-SQ_C_delta,EEG_SQ_P-SQ_C_theta,EEG_SQ_P-SQ_C_alpha,EEG_SQ_P-SQ_C_beta,EEG_SQ_P-SQ_C_gamma,label
0,0.042017,1.128411,1.062267,2.508635,1.242491,6.019582,0.202016,0.036808,0.010602,0.029822,...,3.734651,2.539744,1.209874,21.696838,2.291484,0.442268,0.130891,0.353151,6.767851,0
1,-0.000456,2.064470,1.436826,-0.504070,0.207045,6.956245,0.348325,0.022491,0.010127,0.024041,...,5.188082,-0.523314,0.164764,24.568354,5.037287,0.318157,0.114835,0.243502,22.945826,0
2,-0.019056,0.169068,0.411178,3.111696,0.065338,3.652229,0.099385,0.015333,0.005798,0.021619,...,1.471186,2.991032,0.164141,13.353189,1.263985,0.163334,0.080147,0.301024,0.153148,0
